# FormFactor 財務分析 — Step 2：CSV → SQL 資料庫 → 財務比率分析

上一步我們得到乾淨的 CSV。這一步要：

| 步驟 | 做什麼 | 學到的 SQL |
|---|---|---|
| 1 | 建立 SQLite 資料庫、設計資料表 | `CREATE TABLE`、`PRIMARY KEY` |
| 2 | 把 CSV 載入資料庫 | `INSERT` |
| 3 | 基本查詢 | `SELECT`、`WHERE`、`ORDER BY` |
| 4 | 長表 → 寬表（樞紐） | `CASE WHEN` + `GROUP BY` ★ 面試常考 |
| 5 | 建立「檢視表」 | `CREATE VIEW` |
| 6 | 計算財務比率 | 算術運算、`NULLIF` 防除以零 |
| 7 | 年成長率、平均值 | 視窗函數 `LAG() OVER (...)` ★ 面試常考 |
| 8 | 多步驟查詢 | `WITH`（CTE） |
| 9 | 輸出結果 | 存回 CSV |

> **為什麼用 SQLite？** 它是一個「檔案型」資料庫：整個資料庫就是一個 `.db` 檔，不用安裝伺服器，Python 內建就有。
> 語法和公司常用的 MySQL / PostgreSQL / Snowflake 有 95% 相同，學會就能直接帶去工作。

## 0. 環境設定

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

BASE = Path.cwd()
CLEAN_DIR = BASE / "data" / "clean"
OUT_DIR = BASE / "data" / "analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = BASE / "data" / "formfactor.db"

print("SQLite 版本:", sqlite3.sqlite_version)   # 視窗函數需要 3.25 以上

SQLite 版本: 3.51.0


## 1. 建立資料庫和資料表

`sqlite3.connect()` 會打開資料庫檔案，**檔案不存在就自動建立**。

接著用 `CREATE TABLE` 設計資料表。設計資料表時要決定三件事：
1. **欄位名稱**
2. **資料型態**：`TEXT` 文字、`INTEGER` 整數、`REAL` 小數
3. **主鍵 (PRIMARY KEY)**：哪些欄位組合起來能唯一識別一列。這裡是 (股票代號, 年度, 指標)，同一家公司同一年的營收只能有一筆。資料庫會幫你擋掉重複資料，這就是 Step 1 去重的「最後一道防線」。

In [9]:
conn = sqlite3.connect(DB_PATH)


# 小工具：執行 SQL 查詢，結果直接變成 DataFrame
def q(sql):
    return pd.read_sql_query(sql, conn)

conn.executescript('''
DROP TABLE IF EXISTS financials;
DROP TABLE IF EXISTS metric_dictionary;

CREATE TABLE financials (
    ticker       TEXT    NOT NULL,
    fiscal_year  INTEGER NOT NULL,
    statement    TEXT    NOT NULL,   -- IS 損益表 / BS 資產負債表 / CF 現金流量表
    metric       TEXT    NOT NULL,
    value        REAL,
    concept      TEXT,               -- 原始 XBRL 標籤
    period_end   TEXT,
    filed_date   TEXT,
    accn         TEXT,
    PRIMARY KEY (ticker, fiscal_year, metric)
);

CREATE TABLE metric_dictionary (
    metric           TEXT PRIMARY KEY,
    statement        TEXT,
    xbrl_candidates  TEXT
);
''')
print("✅ 資料表建立完成")

✅ 資料表建立完成


## 2. 把 CSV 載入資料庫

`df.to_sql(..., if_exists="append")` 會把 DataFrame 的每一列轉成 `INSERT INTO ...` 指令，塞進我們剛剛設計好的表。

> 為什麼不直接用 `if_exists="replace"`？因為 replace 會丟掉我們設計的表，讓 pandas 自己猜欄位型態，主鍵也會消失。**先設計表、再塞資料**是正確做法。

In [10]:
fin = pd.read_csv(CLEAN_DIR / "formfactor_financials_long.csv")
dic = pd.read_csv(CLEAN_DIR / "metric_dictionary.csv")
fin.head()
fin.shape
fin.to_sql("financials", conn, if_exists="append", index=False)
dic.to_sql("metric_dictionary", conn, if_exists="append", index=False)
conn.commit()   # commit = 確認寫入（像按下「儲存」）

q("SELECT COUNT(*) AS 總列數, COUNT(DISTINCT metric) AS 指標數, MIN(fiscal_year) AS 起始年, MAX(fiscal_year) AS 結束年 FROM financials")

,總列數,指標數,起始年,結束年
0,278,28,2016,2025


## 3. 基本查詢：SELECT / WHERE / ORDER BY

SQL 的語法像一句英文：
```sql
SELECT   要哪些欄位
FROM     從哪張表
WHERE    篩選條件
ORDER BY 排序
```

In [11]:
# 營收逐年變化（換算成百萬美元，ROUND 到小數 1 位）
q('''
SELECT fiscal_year,
       ROUND(value / 1e6, 1) AS revenue_musd
FROM   financials
WHERE  metric = 'revenue'
ORDER  BY fiscal_year
''')

,fiscal_year,revenue_musd
0,2016,383.90
1,2017,548.40
2,2018,529.70
3,2019,589.50
4,2020,693.60
5,2021,769.70
6,2022,747.90
7,2023,663.10
8,2024,763.60
9,2025,785.00


In [12]:
# IN：一次篩多個指標；看最近一年的損益表
q('''
SELECT metric, ROUND(value / 1e6, 1) AS musd
FROM   financials
WHERE  fiscal_year = 2025
  AND  metric IN ('revenue', 'gross_profit', 'rd_expense', 'sga_expense', 'operating_income', 'net_income')
ORDER  BY value DESC
''')

,metric,musd
0,revenue,785.00
1,gross_profit,308.90
2,sga_expense,133.10
3,rd_expense,115.70
4,operating_income,57.10
5,net_income,54.40


In [13]:
# JOIN：把指標字典接上來，看每個指標用的是哪個 XBRL 標籤
q('''
SELECT f.metric, d.statement, f.concept AS used_tag, d.xbrl_candidates
FROM   financials f
JOIN   metric_dictionary d ON f.metric = d.metric
WHERE  f.fiscal_year = 2025
ORDER  BY d.statement, f.metric
''')

,metric,statement,used_tag,xbrl_candidates
0,accounts_receivable,BS,AccountsReceivableNetCurrent,AccountsReceivableNetCurrent
1,cash,BS,CashAndCashEquivalentsAtCarryingValue,CashAndCashEquivalentsAtCarryingValue
2,current_assets,BS,AssetsCurrent,AssetsCurrent
3,current_liabilities,BS,LiabilitiesCurrent,LiabilitiesCurrent
4,goodwill,BS,Goodwill,Goodwill
5,inventory,BS,InventoryNet,InventoryNet
6,liab_and_equity,BS,LiabilitiesAndStockholdersEquity,LiabilitiesAndStockholdersEquity
7,long_term_debt,BS,LongTermDebtNoncurrent,LongTermDebtNoncurrent | LongTermDebt
8,st_investments,BS,AvailableForSaleSecuritiesDebtSecuritiesCurrent,MarketableSecuritiesCurrent | AvailableForSale...
9,total_assets,BS,Assets,Assets


## 4. 長表 → 寬表：`CASE WHEN` 樞紐 ★

我們的表是「長的」（每列一個數字）。但算比率時需要「寬的」（一年一列，營收、淨利並排），才能寫 `net_income / revenue`。

技巧：`MAX(CASE WHEN metric = 'revenue' THEN value END)`
- `CASE WHEN`：如果這一列是營收就取它的值，不是就給 NULL
- `GROUP BY fiscal_year`：每年壓成一列
- `MAX()`：每年只有一個非 NULL 值，MAX 就把它挑出來

> 這是**資料分析面試最常考的 SQL 題型之一**，一定要會。

In [14]:
q('''
SELECT fiscal_year,
       MAX(CASE WHEN metric = 'revenue'    THEN value END) / 1e6 AS revenue,
       MAX(CASE WHEN metric = 'net_income' THEN value END) / 1e6 AS net_income
FROM   financials
GROUP  BY fiscal_year
ORDER  BY fiscal_year
''')

,fiscal_year,revenue,net_income
0,2016,383.88,-6.56
1,2017,548.44,40.91
2,2018,529.67,104.04
3,2019,589.46,39.35
4,2020,693.62,78.52
5,2021,769.67,83.92
6,2022,747.94,50.74
7,2023,663.10,82.39
8,2024,763.60,69.61
9,2025,784.99,54.36


## 5. 建立檢視表 (VIEW)

每次都寫一長串 `CASE WHEN` 很累。`CREATE VIEW` 可以把一個查詢**存成一張「虛擬表」**，之後直接 `SELECT * FROM fin_wide` 就好。

VIEW 不會複製資料，每次查詢時才即時計算，所以原始資料更新後 VIEW 也會自動更新。

下面用 Python 自動產生 28 個指標的 `CASE WHEN`，不用手打：

In [15]:
metrics = q("SELECT DISTINCT metric FROM financials ORDER BY metric")["metric"].tolist()

case_lines = ",\n       ".join(
    f"MAX(CASE WHEN metric = '{m}' THEN value END) AS {m}" for m in metrics
)
view_sql = f'''
CREATE VIEW fin_wide AS
SELECT ticker, fiscal_year,
       {case_lines}
FROM   financials
GROUP  BY ticker, fiscal_year
'''
conn.execute("DROP VIEW IF EXISTS fin_wide")
conn.execute(view_sql)
print(view_sql[:600], "...")   # 印出前面一段，看看產生的 SQL 長什麼樣子


CREATE VIEW fin_wide AS
SELECT ticker, fiscal_year,
       MAX(CASE WHEN metric = 'accounts_receivable' THEN value END) AS accounts_receivable,
       MAX(CASE WHEN metric = 'acquisitions' THEN value END) AS acquisitions,
       MAX(CASE WHEN metric = 'buybacks' THEN value END) AS buybacks,
       MAX(CASE WHEN metric = 'capex' THEN value END) AS capex,
       MAX(CASE WHEN metric = 'cash' THEN value END) AS cash,
       MAX(CASE WHEN metric = 'cost_of_revenue' THEN value END) AS cost_of_revenue,
       MAX(CASE WHEN metric = 'current_assets' THEN value END) AS current_assets,
       MAX(CASE ...


In [16]:
q("SELECT fiscal_year, revenue, gross_profit, net_income, total_equity FROM fin_wide")

,fiscal_year,revenue,gross_profit,net_income,total_equity
0,2016,"383,881,000.00","102,682,000.00","-6,557,000.00","401,056,000.00"
1,2017,"548,441,000.00","215,597,000.00","40,913,000.00","458,637,000.00"
2,2018,"529,675,000.00","210,339,000.00","104,036,000.00","580,164,000.00"
3,2019,"589,464,000.00","237,496,000.00","39,346,000.00","640,997,000.00"
4,2020,"693,616,000.00","287,920,000.00","78,523,000.00","744,084,000.00"
5,2021,"769,674,000.00","322,767,000.00","83,924,000.00","815,780,000.00"
6,2022,"747,937,000.00","296,009,000.00","50,738,000.00","808,285,000.00"
7,2023,"663,102,000.00","258,580,000.00","82,387,000.00","908,804,000.00"
8,2024,"763,599,000.00","307,923,000.00","69,614,000.00","947,768,000.00"
9,2025,"784,993,000.00","308,851,000.00","54,361,000.00","1,035,403,000.00"


## 6. 財務比率

| 類別 | 比率 | 公式 | 看什麼 |
|---|---|---|---|
| 獲利能力 | 毛利率 | 毛利 ÷ 營收 | 產品定價能力、技術壁壘 |
| | 營業利益率 | 營業利益 ÷ 營收 | 本業賺錢效率 |
| | 淨利率 | 淨利 ÷ 營收 | 最後真正留下多少 |
| 投入 | 研發費用率 | 研發 ÷ 營收 | 對技術的投資強度（半導體設備業很重要） |
| 效率 | ROA | 淨利 ÷ 總資產 | 資產運用效率 |
| 償債 | 流動比率 | 流動資產 ÷ 流動負債 | 短期償債能力，> 1.5 通常算健康 |
| | 負債權益比 | 總負債 ÷ 股東權益 | 財務槓桿 |
| 現金 | 自由現金流 FCF | 營業現金流 − 資本支出 | 真正能自由運用的現金 |
| | 淨現金 | 現金 + 短期投資 − 長期負債 | 手上的「彈藥」 |
| 品質 | SBC 占營收 | 股票酬勞 ÷ 營收 | 對股東的稀釋成本 |

**`NULLIF(x, 0)`**：如果 x 是 0 就變成 NULL。因為除以 0 會出錯，除以 NULL 只會得到 NULL，是 SQL 防呆的標準寫法。

In [17]:
conn.execute("DROP VIEW IF EXISTS ratios")
conn.execute('''
CREATE VIEW ratios AS
SELECT
    ticker,
    fiscal_year,
    revenue / 1e6                                          AS revenue_musd,
    -- 獲利能力（乘 100 變百分比）
    100.0 * gross_profit     / NULLIF(revenue, 0)          AS gross_margin_pct,
    100.0 * operating_income / NULLIF(revenue, 0)          AS operating_margin_pct,
    100.0 * net_income       / NULLIF(revenue, 0)          AS net_margin_pct,
    -- 投入
    100.0 * rd_expense       / NULLIF(revenue, 0)          AS rd_pct_revenue,
    100.0 * sga_expense      / NULLIF(revenue, 0)          AS sga_pct_revenue,
    -- 效率
    100.0 * net_income       / NULLIF(total_assets, 0)     AS roa_pct,
    -- 償債
    current_assets           / NULLIF(current_liabilities, 0) AS current_ratio,
    total_liabilities        / NULLIF(total_equity, 0)     AS debt_to_equity,
    -- 現金
    (operating_cash_flow - capex) / 1e6                    AS fcf_musd,
    100.0 * (operating_cash_flow - capex) / NULLIF(revenue, 0) AS fcf_margin_pct,
    (cash + COALESCE(st_investments, 0) - COALESCE(long_term_debt, 0)) / 1e6 AS net_cash_musd,
    100.0 * capex            / NULLIF(revenue, 0)          AS capex_pct_revenue,
    -- 品質
    100.0 * stock_comp       / NULLIF(revenue, 0)          AS sbc_pct_revenue
FROM fin_wide
''')
q("SELECT * FROM ratios ORDER BY fiscal_year").round(1)

,ticker,fiscal_year,revenue_musd,gross_margin_pct,operating_margin_pct,net_margin_pct,rd_pct_revenue,sga_pct_revenue,roa_pct,current_ratio,debt_to_equity,fcf_musd,fcf_margin_pct,net_cash_musd,capex_pct_revenue,sbc_pct_revenue
0,FORM,2016,383.90,26.70,-12.50,-1.70,15.00,19.10,-1.10,3.10,0.50,NaN,NaN,-24.10,3.00,3.00
1,FORM,2017,548.40,39.30,8.40,7.50,13.50,17.40,6.30,3.30,0.40,68.60,12.50,52.90,3.20,3.00
2,FORM,2018,529.70,39.70,6.80,19.60,14.20,18.70,14.30,3.30,0.30,48.80,9.20,114.00,3.80,3.40
3,FORM,2019,589.50,40.30,8.40,6.70,13.80,18.00,4.70,3.10,0.30,100.20,17.00,205.20,3.50,3.90
4,FORM,2020,693.60,41.50,12.10,11.30,12.80,16.60,8.20,3.20,0.30,113.40,16.30,230.10,8.10,3.40
5,FORM,2021,769.70,41.90,12.70,10.90,13.10,16.10,8.20,3.50,0.30,72.90,9.50,260.60,8.60,3.80
6,FORM,2022,747.90,39.60,7.30,6.80,14.60,17.60,5.00,3.20,0.20,66.50,8.90,223.70,8.70,4.20
7,FORM,2023,663.10,39.00,12.50,12.40,17.50,20.10,7.40,4.40,0.20,8.60,1.30,315.00,8.40,5.80
8,FORM,2024,763.60,40.30,8.50,9.10,16.00,18.60,6.10,4.60,0.20,79.10,10.40,347.80,5.00,5.20
9,FORM,2025,785.00,39.30,7.30,6.90,14.70,17.00,4.40,4.50,0.20,11.70,1.50,264.10,13.20,4.90


> **`COALESCE(a, 0)`**：如果 a 是 NULL 就用 0 代替。2016 年沒有短期投資的資料，不加 COALESCE 的話整個淨現金都會變成 NULL。
>
> **注意 2016 年**：FormFactor 在 2016 年中收購了 Cascade Microtech，營收只含併購後的半年，加上併購相關費用，所以 2016 的比率不具代表性。分析趨勢時通常從 2017 年開始看。

## 7. 視窗函數：年成長率 (YoY) 和 ROE ★

一般的 `GROUP BY` 會把多列壓成一列。**視窗函數**則是保留每一列，同時讓你「看到」其他列。

`LAG(revenue) OVER (ORDER BY fiscal_year)` = 依年度排序後，**取上一列的營收**。

```
fiscal_year  revenue  LAG(revenue)
2023         663.1    747.9   ← 拿到 2022 的值
2024         763.6    663.1   ← 拿到 2023 的值
```

應用：
- **YoY 成長率** = (今年 − 去年) ÷ 去年
- **ROE** 要用「平均股東權益」=(期初 + 期末) ÷ 2，期初就是去年的期末 → 也用 LAG

`PARTITION BY ticker`：之後加入競爭對手時，每家公司各自算，不會把 A 公司的去年接到 B 公司身上。

In [18]:
q('''
SELECT
    fiscal_year,
    revenue / 1e6 AS revenue_musd,
    LAG(revenue) OVER w / 1e6 AS prev_revenue_musd,
    100.0 * (revenue - LAG(revenue) OVER w) / LAG(revenue) OVER w      AS revenue_yoy_pct,
    100.0 * (net_income - LAG(net_income) OVER w) / ABS(LAG(net_income) OVER w) AS net_income_yoy_pct,
    100.0 * (rd_expense - LAG(rd_expense) OVER w) / LAG(rd_expense) OVER w    AS rd_yoy_pct,
    -- ROE：淨利 ÷ 平均股東權益
    100.0 * net_income / ((total_equity + LAG(total_equity) OVER w) / 2.0)   AS roe_pct
FROM fin_wide
WINDOW w AS (PARTITION BY ticker ORDER BY fiscal_year)   -- 定義一次，重複使用
ORDER BY fiscal_year
''').round(1)

,fiscal_year,revenue_musd,prev_revenue_musd,revenue_yoy_pct,net_income_yoy_pct,rd_yoy_pct,roe_pct
0,2016,383.90,NaN,NaN,NaN,NaN,NaN
1,2017,548.40,383.90,42.90,724.00,28.50,9.50
2,2018,529.70,548.40,-3.40,154.30,1.60,20.00
3,2019,589.50,529.70,11.30,-62.20,8.70,6.40
4,2020,693.60,589.50,17.70,99.60,9.20,11.30
5,2021,769.70,693.60,11.00,6.90,13.40,10.80
6,2022,747.90,769.70,-2.80,-39.50,8.20,6.20
7,2023,663.10,747.90,-11.30,62.40,6.00,9.60
8,2024,763.60,663.10,15.20,-15.50,5.30,7.50
9,2025,785.00,763.60,2.80,-21.90,-5.10,5.50


In [19]:
q('''
SELECT fiscal_year,
       ROUND(operating_income / 1e6, 1)                                  AS 財報營業利益,
       ROUND((gross_profit - rd_expense - sga_expense) / 1e6, 1)         AS 自算營業利益,
       ROUND((operating_income - (gross_profit - rd_expense - sga_expense)) / 1e6, 1) AS 差額_其他項目
FROM fin_wide
ORDER BY fiscal_year
''')

,fiscal_year,財報營業利益,自算營業利益,差額_其他項目
0,2016,-47.90,-28.20,-19.70
1,2017,46.30,46.30,0.00
2,2018,36.10,36.10,0.00
3,2019,49.70,49.70,0.00
4,2020,83.80,83.80,0.00
5,2021,98.00,98.00,0.00
6,2022,54.90,54.90,0.00
7,2023,82.80,9.80,73.00
8,2024,64.80,44.20,20.60
9,2025,57.10,60.10,-3.00


### 7.1 移動平均：消除單一年度的雜訊

半導體產業有明顯的**景氣循環**（2022–2023 記憶體下行，2024 回升）。看 3 年移動平均比看單一年度更能看出長期趨勢。

`AVG(...) OVER (ORDER BY fiscal_year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` = 今年加前兩年，三年平均。

In [20]:
q('''
SELECT
    fiscal_year,
    ROUND(gross_margin_pct, 1) AS gross_margin_pct,
    ROUND(AVG(gross_margin_pct) OVER (ORDER BY fiscal_year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS gm_3yr_avg,
    ROUND(operating_margin_pct, 1) AS op_margin_pct,
    ROUND(AVG(operating_margin_pct) OVER (ORDER BY fiscal_year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS opm_3yr_avg
FROM ratios
WHERE fiscal_year >= 2017
ORDER BY fiscal_year
''')

,fiscal_year,gross_margin_pct,gm_3yr_avg,op_margin_pct,opm_3yr_avg
0,2017,39.30,39.30,8.40,8.40
1,2018,39.70,39.50,6.80,7.60
2,2019,40.30,39.80,8.40,7.90
3,2020,41.50,40.50,12.10,9.10
4,2021,41.90,41.20,12.70,11.10
5,2022,39.60,41.00,7.30,10.70
6,2023,39.00,40.20,12.50,10.90
7,2024,40.30,39.60,8.50,9.40
8,2025,39.30,39.60,7.30,9.40


In [21]:
# 1. 建立一次性項目表，放進 2023 年的出售利益
conn.executescript('''
CREATE TABLE IF NOT EXISTS one_time_items (
    ticker       TEXT,
    fiscal_year  INTEGER,
    item         TEXT,      -- 項目名稱
    amount       REAL,      -- 金額（美元，正數代表收益）
    source       TEXT,      -- 資料來源，方便日後查證
    PRIMARY KEY (ticker, fiscal_year, item)
);
DELETE FROM one_time_items;
INSERT INTO one_time_items VALUES
    ('FORM', 2023, 'Gain on sale of FRT business', 73.0e6, 'FY2023 Q4 earnings release');
''')
conn.commit()

# 2. 同時算「財報數字」和「排除一次性項目」的營業利益率與 3 年移動平均
q('''
WITH adj AS (
    SELECT ticker, fiscal_year, SUM(amount) AS one_time
    FROM one_time_items
    GROUP BY ticker, fiscal_year
),
base AS (
    SELECT f.fiscal_year,
           100.0 * f.operating_income / f.revenue                              AS opm_reported,
           100.0 * (f.operating_income - COALESCE(a.one_time, 0)) / f.revenue AS opm_adjusted
    FROM fin_wide f
    LEFT JOIN adj a ON f.ticker = a.ticker AND f.fiscal_year = a.fiscal_year
    WHERE f.fiscal_year >= 2017
)
SELECT fiscal_year,
       ROUND(opm_reported, 1)              AS opm_reported,
       ROUND(AVG(opm_reported) OVER w, 1)  AS opm_3yr_reported,
       ROUND(opm_adjusted, 1)              AS opm_adjusted,
       ROUND(AVG(opm_adjusted) OVER w, 1)  AS opm_3yr_adjusted
FROM base
WINDOW w AS (ORDER BY fiscal_year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
ORDER BY fiscal_year
''')

,fiscal_year,opm_reported,opm_3yr_reported,opm_adjusted,opm_3yr_adjusted
0,2017,8.40,8.40,8.40,8.40
1,2018,6.80,7.60,6.80,7.60
2,2019,8.40,7.90,8.40,7.90
3,2020,12.10,9.10,12.10,9.10
4,2021,12.70,11.10,12.70,11.10
5,2022,7.30,10.70,7.30,10.70
6,2023,12.50,10.90,1.50,7.20
7,2024,8.50,9.40,8.50,5.80
8,2025,7.30,9.40,7.30,5.70


## 8. CTE (`WITH`)：分步驟寫複雜查詢

當查詢變複雜時，用 `WITH 名稱 AS (...)` 把它拆成幾個步驟，像在寫草稿：

**問題：FormFactor 2017→2025 的營收年複合成長率 (CAGR) 是多少？跟研發費用的成長比起來如何？**

CAGR = (期末 ÷ 期初)^(1 ÷ 年數) − 1

> SQLite 沒有內建 `POWER()` 函數（MySQL / PostgreSQL 有），所以我們用 SQL 取出期初期末，最後一步交給 Python 算次方。實務上常常是 **SQL 負責撈資料、Python 負責運算**。

In [22]:
cagr_df = q('''
WITH start_yr AS (
    SELECT * FROM fin_wide WHERE fiscal_year = 2017
),
end_yr AS (
    SELECT * FROM fin_wide WHERE fiscal_year = (SELECT MAX(fiscal_year) FROM fin_wide)
)
SELECT
    s.fiscal_year AS start_year,  e.fiscal_year AS end_year,
    s.revenue    AS rev_start,    e.revenue    AS rev_end,
    s.rd_expense AS rd_start,     e.rd_expense AS rd_end,
    
    s.net_income AS ni_start,     e.net_income AS ni_end,
    s.total_equity AS eq_start,   e.total_equity AS eq_end
FROM start_yr s CROSS JOIN end_yr e
''')

r = cagr_df.iloc[0]
n = r.end_year - r.start_year
for label, a, b in [("營收", r.rev_start, r.rev_end), ("研發費用", r.rd_start, r.rd_end),
                    ("淨利", r.ni_start, r.ni_end), ("股東權益", r.eq_start, r.eq_end)]:
    print(f"{label:6s} CAGR {int(r.start_year)}→{int(r.end_year)}: {((b / a) ** (1 / n) - 1) * 100:5.1f}%")

營收     CAGR 2017→2025:   4.6%
研發費用   CAGR 2017→2025:   5.8%
淨利     CAGR 2017→2025:   3.6%
股東權益   CAGR 2017→2025:  10.7%


### 8.1 景氣循環比較：用 `CASE WHEN` 分期 + `GROUP BY`

把年份分成幾個階段，比較各階段的平均表現：

In [23]:
q('''
WITH tagged AS (
    SELECT *,
           CASE
               WHEN fiscal_year BETWEEN 2017 AND 2019 THEN '1. 2017-19 疫情前'
               WHEN fiscal_year BETWEEN 2020 AND 2022 THEN '2. 2020-22 晶片熱潮'
               ELSE                                       '3. 2023-25 下行與 AI 回升'
           END AS phase
    FROM ratios
    WHERE fiscal_year >= 2017
)
SELECT phase,
       ROUND(AVG(revenue_musd), 0)         AS avg_revenue_musd,
       ROUND(AVG(gross_margin_pct), 1)     AS avg_gm,
       ROUND(AVG(operating_margin_pct), 1) AS avg_opm,
       ROUND(AVG(rd_pct_revenue), 1)       AS avg_rd_pct,
       ROUND(AVG(capex_pct_revenue), 1)    AS avg_capex_pct,
       ROUND(SUM(fcf_musd), 0)             AS total_fcf_musd
FROM tagged
GROUP BY phase
ORDER BY phase
''')

,phase,avg_revenue_musd,avg_gm,avg_opm,avg_rd_pct,avg_capex_pct,total_fcf_musd
0,1. 2017-19 疫情前,556.00,39.80,7.90,13.80,3.50,218.00
1,2. 2020-22 晶片熱潮,737.00,41.00,10.70,13.50,8.50,253.00
2,3. 2023-25 下行與 AI 回升,737.00,39.60,9.40,16.10,8.90,99.00


### 8.2 資本配置：賺到的現金去哪了？

投資分析一定要看**管理層怎麼花錢**：再投資（資本支出、研發）、併購、還是回饋股東（買回庫藏股）？

In [24]:
q('''
SELECT fiscal_year,
       ROUND(operating_cash_flow / 1e6, 1) AS ocf,
       ROUND(capex        / 1e6, 1)        AS capex,
       ROUND(acquisitions / 1e6, 1)        AS acquisitions,
       ROUND(buybacks     / 1e6, 1)        AS buybacks,
       ROUND(100.0 * (capex + acquisitions + buybacks) / NULLIF(operating_cash_flow, 0), 0) AS pct_of_ocf_deployed
FROM fin_wide
WHERE fiscal_year >= 2017
ORDER BY fiscal_year
''')

,fiscal_year,ocf,capex,acquisitions,buybacks,pct_of_ocf_deployed
0,2017,86.30,17.80,0.00,19.00,43.00
1,2018,68.70,19.90,0.00,0.00,29.00
2,2019,121.00,20.80,20.50,0.00,34.00
3,2020,169.30,55.90,51.90,0.00,64.00
4,2021,139.40,66.50,0.00,24.00,65.00
5,2022,131.80,65.30,3.40,82.30,115.00
6,2023,64.60,56.00,0.00,19.80,117.00
7,2024,117.50,38.40,0.00,53.30,78.00
8,2025,115.40,103.70,20.60,26.20,130.00


## 9. 輸出結果

In [25]:
final = q('''
SELECT r.*,
       100.0 * (r.revenue_musd - LAG(r.revenue_musd) OVER w) / LAG(r.revenue_musd) OVER w AS revenue_yoy_pct,
       100.0 * f.net_income / ((f.total_equity + LAG(f.total_equity) OVER w) / 2.0)        AS roe_pct
FROM ratios r
JOIN fin_wide f ON r.ticker = f.ticker AND r.fiscal_year = f.fiscal_year
WINDOW w AS (PARTITION BY r.ticker ORDER BY r.fiscal_year)
ORDER BY r.fiscal_year
''')
final.round(2).to_csv(OUT_DIR / "formfactor_ratios.csv", index=False)
print("✅ 已輸出:", OUT_DIR / "formfactor_ratios.csv")
print("✅ 資料庫:", DB_PATH, "（可以用 DB Browser for SQLite 打開來看）")
conn.close()

✅ 已輸出: C:\Users\User\Documents\FormFactor_Analysis\data\analysis\formfactor_ratios.csv
✅ 資料庫: C:\Users\User\Documents\FormFactor_Analysis\data\formfactor.db （可以用 DB Browser for SQLite 打開來看）


## 📝 從數字讀出故事（請先自己看完上面的結果，再對照這裡）

分析師的價值不在算出比率，而在**解讀**。試著回答這些問題，這就是 Step 1 報告「財務面」的骨架：

1. **成長**：營收從 2017 到 2025 的 CAGR 是多少？哪一年衰退？為什麼？（提示：2023 年記憶體產業大幅減產，DRAM 探針卡需求下滑）
2. **獲利品質**：毛利率長期維持在什麼區間？營業利益率在 2021 年後有改善嗎？還是被研發和管銷吃掉？
3. **研發強度**：研發費用率大約多少？半導體設備同業通常在 10–15%，FormFactor 算高還是低？
4. **資本支出**：2025 年資本支出大幅跳升，占營收比例多少？這代表管理層在擴產嗎？（到 10-K 的 MD&A 章節找管理層的說法，這是 Step 1「公司策略」的線索）
5. **財務體質**：負債權益比和淨現金代表什麼？公司有足夠的錢支撐未來的投資或併購嗎？
6. **股東報酬**：ROE 為什麼不高？是獲利率低，還是股東權益太大（現金太多）？
7. **一次性項目**：2018 年淨利 (104M) 遠高於營業利益 (36M)，看看 `income_tax` 那年是多少？這種一次性的稅務利益要排除，才能看出真實的獲利能力

## ✏️ 練習題
1. 寫一個查詢：找出**營業利益率最高的三年**（提示：`ORDER BY ... DESC LIMIT 3`）
2. 用 `LAG` 算**存貨的年增率**，跟營收年增率比較：存貨長得比營收快通常是警訊
3. 計算**應收帳款週轉天數 (DSO)** = 應收帳款 ÷ 營收 × 365，看看收錢速度有沒有變慢

**下一步 → Step 3 市場與同業分析**：抓 Teradyne、Cohu、Advantest 等同業資料放進同一個資料庫，用 `PARTITION BY ticker` 做同業比較，並研究 FormFactor 的上下游與市占率。